## Монте-Карло для $\pi$: CPU и GPU

### Задание (из `pi_monte_carlo.pdf`)
По заданному числу точек **N** сгенерировать случайные точки в единичном квадрате и оценить $\pi$ как долю точек внутри четверти единичной окружности ($x^2 + y^2 < 1$). Вывести **время выполнения** и **значения $\pi$** для CPU и GPU (результаты могут отличаться).

### Реализованный метод
1. Сгенерировать случайные координаты $X$, $Y$
2. Вычислить $V = X^2 + Y^2$
3. Подсчитать точки, для которых $V < 1$
4. Выполнить редукцию (суммирование)
5. Вычислить $\pi \approx 4 \cdot \frac{count}{N}$

In [2]:
from __future__ import annotations

import math
import time
from dataclasses import dataclass
from statistics import mean
from typing import Any, Optional

import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None


@dataclass(frozen=True)
class EstimateResult:
    backend: str
    n: int
    pi: float
    seconds: float


def _timeit(fn):
    t0 = time.perf_counter()
    out = fn()
    t1 = time.perf_counter()
    return out, (t1 - t0)


def estimate_pi_numpy(n: int, seed: int = 123) -> EstimateResult:
    if n <= 0:
        raise ValueError('N должно быть больше 0')

    rng = np.random.default_rng(seed)

    def run() -> float:
        x = rng.random(n, dtype=np.float64)
        y = rng.random(n, dtype=np.float64)
        inside = (x * x + y * y) < 1.0
        return 4.0 * float(np.count_nonzero(inside)) / float(n)

    pi, sec = _timeit(run)
    return EstimateResult(backend='ЦПУ (NumPy)', n=n, pi=float(pi), seconds=float(sec))


def try_import_cupy() -> Optional[Any]:
    try:
        import cupy as cp  # type: ignore
        _ = cp.cuda.runtime.getDeviceCount()
        return cp
    except Exception:
        return None


def estimate_pi_cupy(n: int, seed: int = 123, chunk_size: int = 5_000_000) -> EstimateResult:
    if n <= 0:
        raise ValueError('N должно быть больше 0')

    cp = try_import_cupy()
    if cp is None:
        raise RuntimeError('CuPy/CUDA недоступны в текущем окружении')

    cp.random.seed(seed)

    def run() -> float:
        total_inside = 0
        remaining = n

        while remaining > 0:
            m = min(chunk_size, remaining)
            x = cp.random.random(m, dtype=cp.float64)
            y = cp.random.random(m, dtype=cp.float64)
            inside = (x * x + y * y) < 1.0
            total_inside += int(cp.count_nonzero(inside).get())
            remaining -= m

        cp.cuda.Stream.null.synchronize()
        return 4.0 * float(total_inside) / float(n)

    pi, sec = _timeit(run)
    return EstimateResult(backend='ГПУ (CuPy/CUDA)', n=n, pi=float(pi), seconds=float(sec))


def benchmark_backend(backend_fn, n: int, repeats: int, seed: int, **kwargs):
    out = []
    for i in range(repeats):
        out.append(backend_fn(n=n, seed=seed + i, **kwargs))
    return out


def summarize_results(results):
    first = results[0]
    pi_values = [r.pi for r in results]
    times = [r.seconds for r in results]
    pi_mean = mean(pi_values)
    return {
        'backend': first.backend,
        'N': first.n,
        'runs': len(results),
        'pi_mean': pi_mean,
        'abs_error': abs(pi_mean - math.pi),
        'time_mean_s': mean(times),
        'time_min_s': min(times),
        'time_max_s': max(times),
    }

In [3]:
# Входные данные из задания: N (количество точек)
# Можно задать одно значение или список для исследования масштабирования.
N_VALUES = [100_000, 1_000_000, 5_000_000]
REPEATS = 3
SEED = 123

N_VALUES, REPEATS, SEED

([100000, 1000000, 5000000], 3, 123)

In [4]:
cpu_rows = []
gpu_rows = []
gpu_available = try_import_cupy() is not None

for n in N_VALUES:
    cpu_runs = benchmark_backend(estimate_pi_numpy, n=n, repeats=REPEATS, seed=SEED)
    cpu_rows.append(summarize_results(cpu_runs))

    if gpu_available:
        gpu_runs = benchmark_backend(estimate_pi_cupy, n=n, repeats=REPEATS, seed=SEED)
        gpu_rows.append(summarize_results(gpu_runs))

if not gpu_available:
    print('Запуск GPU пропущен: CuPy/CUDA недоступны в текущем окружении')

cpu_rows, gpu_rows[:1]

([{'backend': 'ЦПУ (NumPy)',
   'N': 100000,
   'runs': 3,
   'pi_mean': 3.1398800000000002,
   'abs_error': 0.0017126535897928896,
   'time_mean_s': 0.00546629866666611,
   'time_min_s': 0.0034147419999612794,
   'time_max_s': 0.008230160000039177},
  {'backend': 'ЦПУ (NumPy)',
   'N': 1000000,
   'runs': 3,
   'pi_mean': 3.140856,
   'abs_error': 0.0007366535897932458,
   'time_mean_s': 0.017805240666670368,
   'time_min_s': 0.017044235000014396,
   'time_max_s': 0.01901748900002076},
  {'backend': 'ЦПУ (NumPy)',
   'N': 5000000,
   'runs': 3,
   'pi_mean': 3.1419565333333335,
   'abs_error': 0.0003638797435403518,
   'time_mean_s': 0.09238285633335863,
   'time_min_s': 0.09126031100004184,
   'time_max_s': 0.09458263499999475}],
 [{'backend': 'ГПУ (CuPy/CUDA)',
   'N': 100000,
   'runs': 3,
   'pi_mean': 3.1425866666666664,
   'abs_error': 0.0009940130768733013,
   'time_mean_s': 0.46799155733333464,
   'time_min_s': 0.0006181940000260511,
   'time_max_s': 1.402512139999999}])

In [5]:
all_rows = cpu_rows + gpu_rows

if pd is not None:
    df = pd.DataFrame(all_rows)
    df = df[['backend', 'N', 'runs', 'pi_mean', 'abs_error', 'time_mean_s', 'time_min_s', 'time_max_s']]
    display(df.sort_values(['N', 'backend']).reset_index(drop=True))
else:
    for row in all_rows:
        print(row)

,backend,N,runs,pi_mean,abs_error,time_mean_s,time_min_s,time_max_s
0,ГПУ (CuPy/CUDA),100000,3,3.142587,0.000994,0.467992,0.000618,1.402512
1,ЦПУ (NumPy),100000,3,3.139880,0.001713,0.005466,0.003415,0.008230
2,ГПУ (CuPy/CUDA),1000000,3,3.140848,0.000745,0.002115,0.001889,0.002563
3,ЦПУ (NumPy),1000000,3,3.140856,0.000737,0.017805,0.017044,0.019017
4,ГПУ (CuPy/CUDA),5000000,3,3.141684,0.000092,0.008936,0.008776,0.009233
5,ЦПУ (NumPy),5000000,3,3.141957,0.000364,0.092383,0.091260,0.094583


In [6]:
# Финальный обязательный вывод для одного N из входных данных (берём максимальное N).
N_FINAL = max(N_VALUES)
cpu_final = estimate_pi_numpy(N_FINAL, seed=SEED)

print(f'{cpu_final.backend}: pi={cpu_final.pi:.10f}, время={cpu_final.seconds:.6f}с, N={cpu_final.n}')

try:
    gpu_final = estimate_pi_cupy(N_FINAL, seed=SEED)
    print(f'{gpu_final.backend}: pi={gpu_final.pi:.10f}, время={gpu_final.seconds:.6f}с, N={gpu_final.n}')
except Exception as exc:
    print('Финальный запуск GPU пропущен:', exc)

print(f'Эталон math.pi: {math.pi:.10f}')

ЦПУ (NumPy): pi=3.1414608000, время=0.087799с, N=5000000
ГПУ (CuPy/CUDA): pi=3.1416312000, время=0.008928с, N=5000000
Эталон math.pi: 3.1415926536
